# Backend API Tester — MTS AI Hub
Все эндпоинты + замер времени ответа. Бэкенд: `http://localhost:8000`

In [122]:
import httpx, json, struct, io, time
from pathlib import Path

BASE = "http://localhost:8000"
USER_ID = "test_user"
TIMEOUT = 300  # 5 минут для тяжёлых моделей

def ok(label, resp, elapsed=None):
    icon = '✅' if resp.status_code < 400 else '❌'
    t = f' ({elapsed:.1f}s)' if elapsed else ''
    print(f"{icon} [{resp.status_code}] {label}{t}")
    try: print(json.dumps(resp.json(), ensure_ascii=False, indent=2)[:600])
    except: print(resp.text[:400])
    print()

def timed_get(url, **kw):
    t0 = time.time()
    r = httpx.get(url, timeout=TIMEOUT, **kw)
    return r, time.time() - t0

def timed_post(url, **kw):
    t0 = time.time()
    r = httpx.post(url, timeout=TIMEOUT, **kw)
    return r, time.time() - t0

# Создаём тестовые файлы
Path('/tmp/test_doc.txt').write_text('Это тестовый документ. Python — мощный язык. FastAPI быстрый.', encoding='utf-8')

from docx import Document
doc = Document()
doc.add_heading('Тестовый документ', level=1)
doc.add_paragraph('Python — мощный язык. MTS AI Hub — платформа для ИИ.')
doc.save('/tmp/test_doc.docx')

pdf = b'%PDF-1.4\n1 0 obj<</Type/Catalog/Pages 2 0 R>>endobj\n2 0 obj<</Type/Pages/Kids[3 0 R]/Count 1>>endobj\n3 0 obj<</Type/Page/Parent 2 0 R/MediaBox[0 0 612 792]/Contents 4 0 R/Resources<</Font<</F1 5 0 R>>>>>>endobj\n4 0 obj<</Length 44>>stream\nBT /F1 12 Tf 72 700 Td (Test PDF) Tj ET\nendstream\nendobj\n5 0 obj<</Type/Font/Subtype/Type1/BaseFont/Helvetica>>endobj\nxref\n0 6\n0000000000 65535 f \ntrailer<</Size 6/Root 1 0 R>>\nstartxref\n0\n%%EOF'
Path('/tmp/test_doc.pdf').write_bytes(pdf)

sr = 16000
wav_data = b'\x00\x00' * sr
wav = io.BytesIO()
wav.write(b'RIFF')
wav.write(struct.pack('<I', 36 + len(wav_data)))
wav.write(b'WAVEfmt ')
wav.write(struct.pack('<IHHIIHH', 16, 1, 1, sr, sr*2, 2, 16))
wav.write(b'data')
wav.write(struct.pack('<I', len(wav_data)))
wav.write(wav_data)
Path('/tmp/test_audio.wav').write_bytes(wav.getvalue())

print('✅ Файлы: test_doc.txt, test_doc.pdf, test_doc.docx, test_audio.wav')

✅ Файлы: test_doc.txt, test_doc.pdf, test_doc.docx, test_audio.wav


## 1. Health Check

In [123]:
r, t = timed_get(f"{BASE}/v1/health")
ok("Health Check", r, t)

✅ [200] Health Check (0.5s)
{
  "status": "ok",
  "services": {
    "postgres": true,
    "mws_api": true,
    "image_gen": "media-service (SD)",
    "asr": "mws(whisper-turbo-local)"
  }
}



## 2. Chat — Text (mws-gpt-alpha)

In [124]:
r, t = timed_post(f"{BASE}/v1/chat/completions", json={
    "model": "mws-gpt-alpha",
    "messages": [{"role": "user", "content": "Привет! Кто ты?"}],
    "stream": False, "user": USER_ID
})
ok("Chat / Text (mws-gpt-alpha)", r, t)

✅ [200] Chat / Text (mws-gpt-alpha) (3.2s)
{
  "id": "chatcmpl-82ea8ed7ed549954",
  "created": 1775980412,
  "model": "mws-gpt-alpha",
  "object": "chat.completion",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "Привет! Я - MWS GPT, большая языковая модель, основанная на архитектуре GPT. Я могу отвечать на вопросы, давать информацию, помогать с задачами и т.п. Я всегда готов помочь и ответить на любой вопрос, который вы мне зададите.",
        "role": "assistant"
      }
    }
  ],
  "usage": {
    "completion_tokens": 73,
    "prompt_tokens": 299,
    "total_tokens": 372
 



## 3. Chat — Code (qwen3-coder-480b-a35b)

In [125]:
r, t = timed_post(f"{BASE}/v1/chat/completions", json={
    "model": "qwen3-coder-480b-a35b",
    "messages": [{"role": "user", "content": "Напиши функцию сортировки пузырьком на Python"}],
    "stream": False, "user": USER_ID
})
ok("Chat / Code (qwen3-coder-480b-a35b)", r, t)

✅ [200] Chat / Code (qwen3-coder-480b-a35b) (10.6s)
{
  "id": "chatcmpl-66c5e95d6a0b4c1fad59204822d9d6c7",
  "created": 1775980414,
  "model": "qwen3-coder-480b-a35b",
  "object": "chat.completion",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "Вот простая реализация сортировки пузырьком на Python:\n\n```python\ndef bubble_sort(arr):\n    n = len(arr)\n    # Проходим по всем элементам массива\n    for i in range(n):\n        # Флаг для оптимизации: если не было обменов, значит массив отсортирован\n        swapped = False\n        # Последние i элементов уже на своих местах\n        



## 4. Chat — Long Context (qwen2.5-72b-instruct)

In [126]:
r, t = timed_post(f"{BASE}/v1/chat/completions", json={
    "model": "qwen2.5-72b-instruct",
    "messages": [{"role": "user", "content": "Объясни трансформерные архитектуры кратко"}],
    "stream": False, "user": USER_ID
})
ok("Chat / Long Context (qwen2.5-72b-instruct)", r, t)

✅ [200] Chat / Long Context (qwen2.5-72b-instruct) (14.3s)
{
  "id": "chatcmpl-400770bb01044041bffbfd91c05eacac",
  "created": 1775980426,
  "model": "qwen2.5-72b-instruct",
  "object": "chat.completion",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "Трансформеры - это тип архитектуры нейронной сети, который был впервые описан в статье Google 2017 года \"Attention is All You Need\". Изначально они были разработаны для обработки задач обработки естественного языка, но теперь также используются в других областях.\n\nОсновное преимущество трансформеров - это их способность обрабатывать инпуты



## 5. Chat — Auto Router (model=auto, роутинг через MWS)

In [127]:
r, t = timed_post(f"{BASE}/v1/chat/completions", json={
    "model": "auto",
    "messages": [{"role": "user", "content": "Напиши алгоритм бинарного поиска"}],
    "stream": False, "user": USER_ID
})
ok("Chat / Auto Router → код", r, t)

✅ [200] Chat / Auto Router → код (11.5s)
{
  "id": "chatcmpl-dd54934917c34b7dbae161175077bf34",
  "created": 1775980440,
  "model": "qwen3-coder-480b-a35b",
  "object": "chat.completion",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "Вот классическая реализация алгоритма **бинарного поиска** на языке Python:\n\n### Условия:\n- Массив должен быть **отсортирован**.\n- Алгоритм ищет индекс элемента `target` в массиве `arr`, если он присутствует, иначе возвращает `-1`.\n\n---\n\n### 🔍 Итеративная версия:\n\n```python\ndef binary_search(arr, target):\n    left, right = 0, len(



## 6. Chat — Streaming (SSE)

In [128]:
print("🔄 Streaming:")
t0 = time.time()
chunks = []
with httpx.stream("POST", f"{BASE}/v1/chat/completions", json={
    "model": "mws-gpt-alpha",
    "messages": [{"role": "user", "content": "Расскажи короткий анекдот"}],
    "stream": True, "user": USER_ID
}, timeout=TIMEOUT) as resp:
    print(f"Status: {resp.status_code}")
    for line in resp.iter_lines():
        if line.startswith("data: ") and line != "data: [DONE]":
            try:
                delta = json.loads(line[6:])["choices"][0]["delta"].get("content", "")
                if delta: chunks.append(delta); print(delta, end="", flush=True)
            except: pass
t = time.time() - t0
print(f"\n\n✅ chunks: {len(chunks)}, time: {t:.1f}s")

🔄 Streaming:
Status: 200
Почему компьютер попал в психиатрическую клинику? 

Потому что он хранит все проблемы в памяти! (смеёмся)

✅ chunks: 39, time: 2.6s


## 7. Embeddings (bge-m3)

In [129]:
r, t = timed_post(f"{BASE}/v1/embeddings", json={"model": "bge-m3", "input": "Тестовый текст"})
ok("Embeddings (bge-m3)", r, t)

✅ [200] Embeddings (bge-m3) (0.5s)
{
  "model": "/models/bge-m3",
  "data": [
    {
      "embedding": [
        -0.038339663,
        0.04211078,
        -0.0064238342,
        -0.0049958024,
        0.0042633,
        -0.040040363,
        0.010305864,
        0.025565937,
        -0.0187539,
        0.0048525366,
        -0.00010217767,
        -0.0018277889,
        -0.022552742,
        -0.02501136,
        -0.00080413464,
        -0.030464688,
        -0.0029623583,
        -0.00887321,
        -0.036934737,
        -0.019909265,
        -0.038413607,
        -0.0103890505,
        0.024900446,
        0.034993723,
      



## 8. List Models

In [130]:
r, t = timed_get(f"{BASE}/v1/models")
ok("List Models", r, t)

✅ [200] List Models (0.9s)
{
  "data": [
    {
      "id": "deepseek-r1-distill-qwen-32b",
      "object": "model",
      "created": 1677610602,
      "owned_by": "openai"
    },
    {
      "id": "gpt-oss-20b",
      "object": "model",
      "created": 1677610602,
      "owned_by": "openai"
    },
    {
      "id": "cotype-pro-vl-32b",
      "object": "model",
      "created": 1677610602,
      "owned_by": "openai"
    },
    {
      "id": "llama-3.1-8b-instruct",
      "object": "model",
      "created": 1677610602,
      "owned_by": "openai"
    },
    {
      "id": "QwQ-32B",
      "object": "model",
      "created"



## 9. Deep Research (SSE)

In [131]:
print("🔄 Deep Research:")
t0 = time.time()
with httpx.stream("POST", f"{BASE}/v1/research", json={"query": "Квантовые вычисления", "user_id": USER_ID}, timeout=TIMEOUT) as resp:
    print(f"Status: {resp.status_code}")
    for line in resp.iter_lines():
        if line: print(line[:120])
t = time.time() - t0
print(f"\n✅ Research done ({t:.1f}s)")

🔄 Deep Research:
Status: 200
event: progress
data: {"step": 1, "message": "Генерирую подзапросы…"}
event: progress
data: {"step": 2, "sub_queries": ["Шуты Кнута и квантовые вычисления", "Применение квантовых вычислений в криптографии",
event: progress
data: {"step": 3, "message": "Ищу и парсю страницы…"}
event: progress
data: {"step": 4, "pages_fetched": 15}
event: progress
data: {"step": 5, "message": "Синтезирую ответ…"}
event: done
data: {"answer": "Квантовые вычисления – это революционный подход к обработке информации, а основой их служат основы ква

✅ Research done (56.3s)


## 10. Web Search

In [132]:
r, t = timed_post(f"{BASE}/v1/tools/web-search", json={"query": "FastAPI Python 2025", "max_results": 3})
ok("Web Search", r, t)

✅ [200] Web Search (4.5s)
{
  "results": [
    {
      "title": "fastapi · PyPI",
      "url": "https://pypi.org/project/fastapi/",
      "snippet": "Apr 1, 2026 ·\" If anyone is looking to build a productionPythonAPI, I would highly recommendFastAPI. It is beautifully designed, simple to use and highly scalable, it has become a key component in ourAPIfirst development strategy and is driving many automations and services such as our Virtual TAC Engineer."
    },
    {
      "title": "FastAPI in 2025: The Modern Python Framework ... - Medium",
      "url": "https://medium.com/pythoneers/fastapi-in-2025-the-modern-pytho



## 11. Web Parse

In [133]:
r, t = timed_post(f"{BASE}/v1/tools/web-parse", json={"url": "https://ru.wikipedia.org/wiki/%D0%92%D0%B8%D0%BA%D0%B8%D0%BF%D0%B5%D0%B4%D0%B8%D1%8F"})
ok("Web Parse", r, t)

✅ [200] Web Parse (1.2s)
{
  "url": "https://ru.wikipedia.org/wiki/%D0%92%D0%B8%D0%BA%D0%B8%D0%BF%D0%B5%D0%B4%D0%B8%D1%8F",
  "text": "Википедия — Википедия\nПерейти к содержанию\nЭта страница защищена от редактирования (частичная защита (до автоподтверждённых)).\nМатериал из Википедии — свободной энциклопедии\nСтабильная версия\n, проверенная 30 марта 2026.\nУ этого термина существуют и другие значения, см.\nВикипедия (значения)\n.\nЭта статья — о многоязычном веб-сайте, являющемся одним из проектов\nWikimedia Foundation\n. О разделах энциклопедии на разных языках см.\nЯзыковые разделы Википедии\n; о данном (русскояз



## 12. File Upload — TXT

In [134]:
with open('/tmp/test_doc.txt', 'rb') as f:
    r, t = timed_post(f"{BASE}/v1/files/upload",
        files={"file": ("test_doc.txt", f, "text/plain")},
        data={"user_id": USER_ID})
ok("File Upload (TXT)", r, t)

✅ [200] File Upload (TXT) (1.1s)
{
  "file_id": "60fb95ac-daa2-44f2-8b1b-a5a43a1351a0",
  "chunks": 1
}



## 13. File Upload — PDF

In [135]:
with open('/tmp/test_doc.pdf', 'rb') as f:
    r, t = timed_post(f"{BASE}/v1/files/upload",
        files={"file": ("test_doc.pdf", f, "application/pdf")},
        data={"user_id": USER_ID})
ok("File Upload (PDF)", r, t)

✅ [200] File Upload (PDF) (0.9s)
{
  "file_id": "9172a716-c236-46c1-82ca-9e37d9f365c9",
  "chunks": 1
}



## 14. File Upload — DOCX

In [136]:
with open('/tmp/test_doc.docx', 'rb') as f:
    r, t = timed_post(f"{BASE}/v1/files/upload",
        files={"file": ("test_doc.docx", f, "application/vnd.openxmlformats-officedocument.wordprocessingml.document")},
        data={"user_id": USER_ID})
ok("File Upload (DOCX)", r, t)

✅ [200] File Upload (DOCX) (1.6s)
{
  "file_id": "a6ca9667-bf90-441e-9529-62f720af9905",
  "chunks": 1
}



## 15. File QA (chat с файлом)

In [137]:
r, t = timed_post(f"{BASE}/v1/chat/completions", json={
    "model": "auto",
    "messages": [{"role": "user", "content": "Что написано в моём документе?"}],
    "stream": False, "user": USER_ID,
    "attachments": [{"name": "test_doc.txt", "mime": "text/plain"}]
})
ok("Chat / File QA", r, t)

✅ [200] Chat / File QA (4.5s)
{
  "id": "chatcmpl-93837a7a22891672",
  "created": 1775980522,
  "model": "mws-gpt-alpha",
  "object": "chat.completion",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "Извините, но я не могу получить доступ к вашему документу, поскольку я не имею возможности видеть или считывать содержимое вашего устройства. Если вы хотите узнать содержимое документа, вы можете описать его мне или предоставить необходимую информацию. \n\nЕсли вы хотите узнать содержимое документа, вы можете ответить на следующие вопросы:\n\n- Какой тип документа у



## 16. Memory — GET

In [138]:
r, t = timed_get(f"{BASE}/v1/memory/{USER_ID}")
ok("Memory GET", r, t)

✅ [200] Memory GET (0.0s)
{
  "user_id": "test_user",
  "memories": []
}



## 17. Memory — Search

In [139]:
r, t = timed_post(f"{BASE}/v1/memory/{USER_ID}/search", json={"query": "Python", "top_k": 5})
ok("Memory Search", r, t)

✅ [200] Memory Search (0.7s)
{
  "user_id": "test_user",
  "results": []
}



## 18. Image Generation (qwen-image через MWS)

In [140]:
r, t = timed_post(f"{BASE}/v1/image/generate", json={
    "prompt": "красивый закат над морем",
    "model": "qwen-image",
    "size": "1024x1024"
})
ok("Image Generate (qwen-image → MWS)", r, t)

✅ [200] Image Generate (qwen-image → MWS) (13.8s)
{
  "created": 1775980525,
  "background": null,
  "data": [
    {
      "b64_json": null,
      "revised_prompt": "sunset",
      "url": "https://imagegen.gpt.mws.ru/files/651888602b76e60a28f43b617a7b3c3ad9b9dd1a4531a501c55de9adff141f3/c6a4355c-6b19-4429-b763-7bf2c2e938bc-0.png"
    }
  ],
  "output_format": "png",
  "quality": "standard",
  "size": "1024x1024",
  "usage": {
    "total_tokens": 767,
    "input_tokens": 2,
    "input_tokens_details": {
      "image_tokens": 0,
      "text_tokens": 2
    },
    "output_tokens": 765
  }
}



## 19. VLM Analyze (JSON — image_url + вопрос)

In [141]:
IMG_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png"

r, t = timed_post(f"{BASE}/v1/vlm/analyze", json={
    "image_url": IMG_URL,
    "question": "Что изображено?",
    "model": "qwen2.5-vl"
})
ok("VLM Analyze", r, t)

✅ [200] VLM Analyze (20.6s)
{
  "answer": null,
  "fallback": true,
  "message": "Server error '500 Internal Server Error' for url 'https://api.gpt.mws.ru/v1/chat/completions'\nFor more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/500"
}



## 20. PPTX Generation

In [142]:
t0 = time.time()
r = httpx.post(f"{BASE}/v1/tools/generate-pptx", json={
    "topic": "Искусственный интеллект",
    "slide_count": 5, "language": "ru"
}, timeout=TIMEOUT)
t = time.time() - t0
if r.status_code == 200 and 'application' in r.headers.get('content-type', ''):
    out = Path('/tmp/test_output.pptx')
    out.write_bytes(r.content)
    print(f"✅ [200] PPTX saved → {out} ({len(r.content):,} bytes) ({t:.1f}s)")
else:
    ok("PPTX Generate", r, t)

❌ [500] PPTX Generate (3.5s)
Internal Server Error



## 21. Voice Message (whisper-turbo-local через MWS)

In [143]:
with open('/tmp/test_audio.wav', 'rb') as f:
    t0 = time.time()
    r = httpx.post(f"{BASE}/v1/voice/message",
        files={"audio": ("test_audio.wav", f, "audio/wav")},
        data={"user_id": USER_ID},
        timeout=TIMEOUT)
    t = time.time() - t0
if r.status_code == 200:
    print(f"✅ [200] Voice pipeline OK ({t:.1f}s)")
    print(f"  Transcript: {r.headers.get('X-Transcript', '?')}")
    print(f"  Answer: {r.headers.get('X-Answer', '?')[:200]}")
    print(f"  Audio: {len(r.content):,} bytes")
elif r.status_code == 503:
    print(f"⚠️  [503] ASR недоступен ({t:.1f}s) — whisper-turbo-local и media-service оба не ответили")
else:
    ok("Voice Message", r, t)

❌ [500] Voice Message (4.6s)
Internal Server Error



## 22. Router Detection — все типы задач (через MWS API)

In [ ]:
cases = [
    ("Привет, как дела?",                          "text",          "mws-gpt-alpha"),
    ("Напиши функцию сортировки на Python",         "code",          "qwen3-coder-480b-a35b"),
    ("Исследуй тему квантовых вычислений",          "deep_research", "qwen2.5-72b-instruct"),
    ("Зайди на https://example.com и расскажи",     "web_parse",     "mws-gpt-alpha"),
    ("Нарисуй закат над морем",                     "image_gen",     "qwen-image"),
    ("Глубокий анализ блокчейна",                  "deep_research", "qwen2.5-72b-instruct"),
    ("Реализуй алгоритм Дейкстры на Go",           "code",          "qwen3-coder-480b-a35b"),
]
print(f"{'Ожид. тип':<18} {'Ожид. модель':<30} {'Время':>7} Статус | Сообщение")
print("-" * 95)
for msg, exp_task, exp_model in cases:
    try:
        t0 = time.time()
        r = httpx.post(f"{BASE}/v1/chat/completions", json={
            "model": "auto",
            "messages": [{"role": "user", "content": msg}],
            "stream": False, "user": USER_ID
        }, timeout=TIMEOUT)
        t = time.time() - t0
        icon = '✅' if r.status_code < 400 else '❌'
        print(f"{exp_task:<18} {exp_model:<30} {t:>6.1f}s {icon} [{r.status_code}] | {msg[:40]}")
    except Exception as e:
        print(f"{exp_task:<18} {exp_model:<30}         ❌ ERR: {e}")

Ожид. тип          Ожид. модель                     Время Статус | Сообщение
-----------------------------------------------------------------------------------------------
text               mws-gpt-alpha                     2.2s ✅ [200] | Привет, как дела?
code               qwen3-coder-480b-a35b            26.9s ✅ [200] | Напиши функцию сортировки на Python
deep_research      qwen2.5-72b-instruct             29.4s ✅ [200] | Исследуй тему квантовых вычислений
web_parse          mws-gpt-alpha                     4.1s ✅ [200] | Зайди на https://example.com и расскажи
image_gen          qwen-image                        2.2s ❌ [404] | Нарисуй закат над морем
deep_research      qwen2.5-72b-instruct             22.4s ✅ [200] | Глубокий анализ блокчейна


## 23. Роутер — детерминированные правила (без API)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.environ.setdefault('MWS_API_KEY', 'test')
os.environ.setdefault('DATABASE_URL', 'postgresql+asyncpg://u:p@localhost/db')
os.environ.setdefault('REDIS_URL', 'redis://localhost:6379/0')

from app.config import get_settings
from app.services.router_client import RouterClient

router = RouterClient(get_settings())

cases = [
    ('Привет, как дела?',                        [],                                          None),
    ('Напиши функцию сортировки на Python',       [],                                          'code'),
    ('Реализуй алгоритм Дейкстры',               [],                                          'code'),
    ('Исследуй тему квантовых вычислений',        [],                                          'deep_research'),
    ('Глубокий анализ блокчейна',                [],                                          'deep_research'),
    ('Зайди на https://example.com',              [],                                          'web_parse'),
    ('',                                          [{'name':'voice.mp3','mime':'audio/mpeg'}],  'asr'),
    ('что на фото?',                              [{'name':'img.jpg','mime':'image/jpeg'}],    'vlm'),
    ('разбери документ',                          [{'name':'doc.pdf','mime':'application/pdf'}],'file_qa'),
    ('',                                          [{'name':'report.docx','mime':'application/vnd.openxmlformats-officedocument.wordprocessingml.document'}], 'file_qa'),
]

print(f"{'Сообщение':<45} {'Ожидается':<15} {'Результат':<15} OK?")
print('-' * 80)
passed = 0
for msg, att, expected in cases:
    result = router._deterministic(msg, att)
    got = result.task_type if result else None
    match = got == expected
    if match: passed += 1
    label = msg or att[0]['name']
    print(f"{label[:44]:<45} {str(expected):<15} {str(got):<15} {'✅' if match else '❌'}")
print(f'\nДетерминированный роутер: {passed}/{len(cases)} ✅')

Сообщение                                     Ожидается       Результат       OK?
--------------------------------------------------------------------------------
Привет, как дела?                             None            None            ✅
Напиши функцию сортировки на Python           code            code            ✅
Реализуй алгоритм Дейкстры                    code            code            ✅
Исследуй тему квантовых вычислений            deep_research   deep_research   ✅
Глубокий анализ блокчейна                     deep_research   deep_research   ✅
Зайди на https://example.com                  web_parse       web_parse       ✅
voice.mp3                                     asr             asr             ✅
что на фото?                                  vlm             vlm             ✅
разбери документ                              file_qa         file_qa         ✅
report.docx                                   file_qa         file_qa         ✅

Детерминированный роутер: 10/10 ✅


## 24. Summary — All Endpoints

In [ ]:
endpoints = [
    ("GET",  "/v1/health",                    None),
    ("POST", "/v1/chat/completions",          {"model":"mws-gpt-alpha","messages":[{"role":"user","content":"hi"}],"stream":False}),
    ("POST", "/v1/embeddings",                {"model":"bge-m3","input":"test"}),
    ("GET",  "/v1/models",                    None),
    ("POST", "/v1/tools/web-search",          {"query":"test","max_results":1}),
    ("POST", "/v1/tools/web-parse",           {"url":"https://example.com"}),
    ("POST", "/v1/tools/generate-pptx",       {"topic":"AI","slide_count":3,"language":"ru"}),
    ("GET",  f"/v1/memory/{USER_ID}",         None),
    ("POST", f"/v1/memory/{USER_ID}/search",  {"query":"test","top_k":3}),
    ("POST", "/v1/image/generate",            {"prompt":"cat","model":"qwen-image","size":"1024x1024"}),
    ("POST", "/v1/vlm/analyze",               {"image_url":"https://example.com/i.png","question":"test","model":"qwen2.5-vl"}),
]
print(f"{'Method':<6} {'Endpoint':<40} {'Status':>6} {'Time':>7}  Result")
print("-" * 70)
passed = 0
for method, path, body in endpoints:
    try:
        t0 = time.time()
        r = httpx.get(f"{BASE}{path}", timeout=TIMEOUT) if method == "GET" else httpx.post(f"{BASE}{path}", json=body, timeout=TIMEOUT)
        t = time.time() - t0
        icon = "✅" if r.status_code < 400 else "⚠️ "
        if r.status_code < 400: passed += 1
        print(f"{method:<6} {path:<40} {r.status_code:>6} {t:>6.1f}s  {icon}")
    except Exception as e:
        print(f"{method:<6} {path:<40} {'ERR':>6}          ❌ {str(e)[:30]}")
print(f"\n{'='*70}\nИтого: {passed}/{len(endpoints)} OK")